## Break RSA!
RSA is used to prove identity and safely exchange secret keys in things like HTTPS websites, digital signatures on software and documents, SSH logins, encrypted email, VPNs, and bank or ID smart cards.

- **$p$ and $q$**: two prime numbers that you keep secret.
- **$N$**: their product (It is public!). Recovering $p$ and $q$ from $N$ is very hard classically when $N$ has hundreds of digits.
- **$e$**: a number you publish together with $N$. To encrypt, a letter becomes a number $m$, and the encrypted value is the remainder of $m^e$ divided by $N$ (aka $m^e\mod N$).
- **$d$**: the number that undoes the encryption. You can only calculate $d$ if you know $p$ and $q$, so we need factoring!

Anyone with $N$ and $e$ can encrypt, but only someone who knows $p$ and $q$ can decrypt.

Shor's algorithm: allows a quantum computer to find $p$ and $q$ from $N$!

In [1]:
N, e = 15, 3
alphabet = "abcdehilmnorstu"

In [2]:
word = "shor"   

#encypt
ciphertext = [pow(alphabet.index(ch), e, N) for ch in word]
print(ciphertext)

[3, 5, 10, 11]


Here will just use the intercepted ciphertext to guess the word

In [3]:
from math import gcd
from fractions import Fraction
from qrisp import QuantumModulus, QuantumFloat, control, QFT, h

##### Shor's Algorithm ######

def shor(a, N, precision):
    target = QuantumModulus(N)
    target[:] = 1                              
    phase = QuantumFloat(precision, exponent=-precision)
    h(phase)                                    
    for i in range(precision):                   
        with control(phase[i]):
            target *= pow(a, 2**i, N)
    QFT(phase, inv=True)                         
    return phase.get_measurement()

##### Let's run it  ######

a = 7
results = shor(a, N, precision=3)

##### Classical Post-proc ######

for ph in results:
    r = Fraction(ph).limit_denominator(N).denominator
    if r % 2 == 0 and pow(a, r, N) == 1:
        break

p = gcd(pow(a, r // 2) - 1, N)
q = gcd(pow(a, r // 2) + 1, N)
d = pow(e, -1, (p - 1) * (q - 1))

##### Show the results ######

print("N =", p, "x", q)
print("Your word was:", "".join(alphabet[pow(c, d, N)] for c in ciphertext))

N = 3 x 5                                                                       
Your word was: shor


### Qrisp's built-in compacted version:

In [4]:
from qrisp.shor import rsa_encrypt_string
intercepted_message=rsa_encrypt_string(e = 11, N = 65 , message = "welcome to the QAlgo course")
intercepted_message

'01101100001000000100100011000010100001010101100100101101010100000011100001100001000000000010001101000110100110010001111011110000010110010110010001001100110110110000110100010110010100001010100110110001000000011100001010101000'

In [5]:
from qrisp.shor import rsa_decrypt_string
rsa_decrypt_string(e = 11, N = 65, ciphertext = intercepted_message)

'welcome to the QAlgo course'